### 1.  BatchNorm2d层（二维批量归一化层）

第一步：计算均值
$$
\mu_B=\frac{1}{m}\sum_{i=1}^{m}x_i
$$

第二步：计算方差
$$
\sigma_B^2=\frac{1}{m}\sum_{i=1}^{m}(x_i-\mu_B)^2
$$

第三步：归一化
$$
\hat{x}_i=\frac{x_i-\mu_B}
{\sqrt{\sigma_B^2+\epsilon}}
$$

经过这一步，数据会变成均值接近 0、方差接近 1 的分布。

第四步：缩放和偏移
$$
y_i=\gamma\hat{x}_i+\beta
$$

In [ ]:
# nn.BatchNorm2d(
#     num_features,
#     eps=1e-5,
#     momentum=0.1,
#     affine=True,
#     track_running_stats=True
# )

#------------------------------#
# num_features中的每层都做归一化处理
# 标准化的过程就是将一个一个数据分布变为均值为0，方差为1的数据分布


|参数|含义|
|---|---|
|num_features|输入特征图的通道数|
|eps|防止除零的小常数|
|momentum|运行均值和方差的更新系数|
|affine|是否启用可学习的 γ 和 β|
|track_running_stats|是否记录运行均值和方差|

### 2.  Sequential层(网络层容器)、AdaptiveAvgPool2d(自适应二维平均池化层)、x = torch.flatten(张量展开层)

In [ ]:
# 1. Sequential可以将一些列的网络层整合到一起，数据会按照顺序向下传递
import torch
import torch.nn as nn

layer1 = nn.Sequential(
    nn.Conv2d(3, 64, kernel_size=3),
    nn.BatchNorm2d(64),
    nn.ReLU()
)
# 2. AdaptiveAvgPool2d无论输入特征图的宽度和高度是多少，都可以将每个通道的特征图压缩成指定大小
layer2 = nn.AdaptiveAvgPool2d((1,1))
# 3.x = torch.flatten(x, 1) 不是一个带参数的“模块”，而是一个张量展平操作，所以它的参数量是 0。它的作用是从第 1 维开始，把后面的所有维度展平，保留第 0 维 batch 维。
x = torch.randn([1,2,3,4])
x = torch.flatten(x, 1)# 从第一维度开始进行展平操作

### 3. 残差块的定义

In [21]:
"""
(B, C, H, W) = (Batch_Size, 3, 224, 224)训练数据
"""
import torch
import torch.nn as nn

class BasicBlock(nn.Module):
    expansion = 1# 卷积层最后输出的通道数与out_channels的商，注意是类属性
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        

        # 第一个卷积层
        self.conv1 = nn.Conv2d(in_channels = in_channels , out_channels = out_channels , stride = stride , kernel_size = 3 , bias = False , padding = 1)
        # 这个模块无法复用
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(in_channels = out_channels , out_channels = out_channels , stride = 1 , kernel_size = 3 , bias = False , padding = 1)

        # 残差连接
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(nn.Conv2d(in_channels,out_channels,1,stride,bias = False),nn.BatchNorm2d(out_channels))
    def forward(self,x):
        # 主分支
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        # 残差连接
        out = self.relu(self.shortcut(x)+out)
        return out


### 4. 构建残差模型

In [22]:
class ResNet(nn.Module):
    def __init__(self, block, num_classes=1000):# num_classes是分类类别
        super().__init__()
        self.in_channels = 64# 这是第一次卷积之后的数字
        self.conv1 = nn.Conv2d(3, 64,kernel_size=7,stride=2,padding=3,bias=False)# (Batch_size,64,224,224)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3,stride=2,padding=1)

        # 四个残差阶段
        self.layer1 = self.make_layer(block, 64, 2, stride=1)
        # 这里后三个残差阶段中只有第一个残差块中的步数变为的了2
        self.layer2 = self.make_layer(block, 128, 2, stride=2)

        self.layer3 = self.make_layer(block, 256, 2, stride=2)

        self.layer4 = self.make_layer(block, 512, 2, stride=2)
        # 全局平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        # 全连接分类层
        self.fc = nn.Linear(512 * block.expansion, num_classes)
    def make_layer(self,block,out_channels,blocks,stride):

        layers = []
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels*block.expansion
        for _ in range(1,blocks):
            layers.append(block(self.in_channels,out_channels))
        return nn.Sequential(*layers)
    def forward(self,x):# (Batch_size,3,112,112)
        # 初始卷积
        x = self.conv1(x)# (Batch_size,64,224,224)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)# (Batch_size,64,112,112)

        # 四个残差阶段
        x = self.layer1(x)# (Batch_size,64,112,112)
        x = self.layer2(x)# (Batch_size,128,56,56)
        x = self.layer3(x)# (Batch_size,256,28,28)
        x = self.layer4(x)# (Batch_size,512,14,14)

        # 平均池化
        x = self.avgpool(x)# (Batch_size,512,1,1)

        # 展平
        x = torch.flatten(x, 1)# (Batch_size,512)

        # 分类
        x = self.fc(x)
        return x

        

### 5. 创建模型

In [23]:
def resnet18(num_classes=1000):

    return ResNet(
        BasicBlock,
        num_classes=num_classes
    )


# 创建模型
model = resnet18(num_classes=1000)

print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=Tru

### 6. 测试尺寸

In [24]:
import torch

model = resnet18(num_classes=10)

# 模拟一批图片
x = torch.randn(1, 3, 224, 224)

# 初始卷积
x = model.conv1(x)
print("conv1:", x.shape)

x = model.bn1(x)
x = model.relu(x)
x = model.maxpool(x)
print("maxpool:", x.shape)

x = model.layer1(x)
print("layer1:", x.shape)

x = model.layer2(x)
print("layer2:", x.shape)

x = model.layer3(x)
print("layer3:", x.shape)

x = model.layer4(x)
print("layer4:", x.shape)

x = model.avgpool(x)
print("avgpool:", x.shape)

x = torch.flatten(x, 1)
print("flatten:", x.shape)

x = model.fc(x)
print("output:", x.shape)

conv1: torch.Size([1, 64, 112, 112])
maxpool: torch.Size([1, 64, 56, 56])
layer1: torch.Size([1, 64, 56, 56])
layer2: torch.Size([1, 128, 28, 28])
layer3: torch.Size([1, 256, 14, 14])
layer4: torch.Size([1, 512, 7, 7])
avgpool: torch.Size([1, 512, 1, 1])
flatten: torch.Size([1, 512])
output: torch.Size([1, 10])
